# 🎙️ Arabic ASR — Audar Turbo vs QwenCleo

**استخدام بسيط جدًا:** اختار GPU، الصق رابط Google Drive، واضغط تشغيل.

> كل منطق التشغيل بيتحمّل تلقائيًا من GitHub عند كل Run، لذلك تحديث المحرك لا يحتاج Notebook جديدة.


In [ ]:
#@title 🎙️ Run Arabic ASR comparison
#@markdown الصق رابط ملف Google Drive ثم اضغط ▶️ فقط.
drive_link = "" #@param {type:"string"}

import json
import re
import subprocess
import time
import urllib.error
import urllib.request

REPO = "abdullahsamirashour/gpt"
BRANCH = "main"
ENGINE_REL = "arabic-asr-benchmark/engine.py"
ENGINE_PATH = "/content/arabic_asr_benchmark_engine.py"

def latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{REPO}.git", f"refs/heads/{BRANCH}"],
        capture_output=True,
        text=True,
        timeout=30,
    )
    if p.returncode == 0 and p.stdout.strip():
        sha = p.stdout.strip().split()[0]
        if re.fullmatch(r"[0-9a-f]{40}", sha):
            return sha

    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}?t={int(time.time())}",
        headers={
            "Accept": "application/vnd.github+json",
            "User-Agent": "Arabic-ASR-Colab",
            "Cache-Control": "no-cache",
        },
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

def download_engine(sha):
    url = f"https://raw.githubusercontent.com/{REPO}/{sha}/{ENGINE_REL}"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Arabic-ASR-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return response.read().decode("utf-8")

if not drive_link.strip():
    raise ValueError("❌ حط رابط Google Drive الأول.")

try:
    print("🔄 Loading latest engine from GitHub...")
    sha = latest_sha()
    ASR_ENGINE_SHA = sha
    engine = download_engine(sha)
    if len(engine) < 4000:
        raise RuntimeError("engine-too-small")
    compiled = compile(engine, ENGINE_PATH, "exec")
    print(f"✅ Loaded commit {sha[:10]}")
    exec(compiled, globals(), globals())
except urllib.error.HTTPError as exc:
    print(f"❌ تعذر الوصول إلى GitHub (HTTP {exc.code}). جرّب مرة أخرى.")
except urllib.error.URLError:
    print("❌ تعذر الاتصال بـ GitHub. تأكد من الإنترنت وجرب مرة أخرى.")
except SyntaxError:
    print("❌ النسخة المنشورة على GitHub فيها خطأ برمجي.")
    raise
